In [1]:
%run "ms_lms_config"

StatementMeta(, 2b0532cd-00ed-45aa-b463-8395221c3157, 4, Finished, Available, Finished)

### 1. Data cleaning

In [43]:
today_date = ''

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 49, Finished, Available, Finished)

In [44]:
df = spark.read.table("ms_lms_lh_100_bronze.lms_100_bronze")

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 50, Finished, Available, Finished)

In [45]:
df = df \
.filter(col('Processing_Date') == today_date)

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 51, Finished, Available, Finished)

#### Handling duplicates

In [46]:
df_deduped = df.dropDuplicates()

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 52, Finished, Available, Finished)

#### Drop missing values

In [47]:
df_droppedna = df_deduped \
.dropna(subset=['Student_ID','Course_ID','Enrollment_Date'])

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 53, Finished, Available, Finished)

#### Fill missing values

In [48]:
df_filledna = df_droppedna.fillna({
                "Age": 0,
                "Gender": "Unknown",
                "Status": "In-progress",
                "Final_Grade": "N/A",
                "Attendance_Rate": 0.0,
                "Time_Spent_on_Course_hrs": 0.0,
                "Assignments_Completed": 0,
                "Quizzes_Completed": 0,
                "Forum_Posts": 0,
                "Messages_Sent": 0,
                "Quiz_Average_Score": 0.0,
                "Assignment_Average_Score": 0.0,
                "Project_Score": 0.0,
                "Extra_Credit": 0.0,
                "Overall_Performance": 0.0,
                "Feedback_Score": 0.0,
                "Parent_Involvement": "Unknown",
                "Demographic_Group": "Unknown",
                "Internet_Access": "Unknown",
                "Learning_Disabilities": "Unknown",
                "Preferred_Learning_Style": "Unknown",
                "Language_Proficiency": "Unknown",
                "Participation_Rate": "Unknown",
                "Completion_Time_Days": 0,
                "Performance_Score": 0.0,
                "Course_Completion_Rate": 0.0,
                "Completion_Date": '12/31/9999'
})

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 54, Finished, Available, Finished)

#### Standarize date formats

In [49]:
df_formated = df_filledna \
                .withColumn("Enrollment_Date", to_date(col('Enrollment_Date'),'M/d/yyyy')) \
                .withColumn("Completion_Date", to_date(col('Completion_Date'),'M/d/yyyy'))

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 55, Finished, Available, Finished)

#### Check for logical consistency

In [50]:
logic_check_cnt = df_formated \
.filter(col('Completion_Date') >= col('Enrollment_Date'))\
.count()

all_cnt = df_formated \
.count()

logic_check_result = logic_check_cnt == all_cnt
display(logic_check_result)

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 56, Finished, Available, Finished)

True

In [51]:
df_consistent = df_formated \
.filter(col('Completion_Date') >= col('Enrollment_Date'))

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 57, Finished, Available, Finished)

### 2. Business transformations

In [53]:
df_business = df_consistent \
    .withColumn('Completion_Time_Days', (col('Completion_Date') - col('Enrollment_Date')).cast('int')) \
    .withColumn('Performance_Score', round(col('Quiz_Average_Score') * 0.2 + col('Assignment_Average_Score') * 0.2 + col('Project_Score') * 0.1, 2)) \
    .withColumn('Course_Completion_Rate', when(col('Completion_Time_Days') <= 90, 'On-Time').otherwise('Delayed'))


StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 59, Finished, Available, Finished)

In [54]:
df_business.createOrReplaceTempView('lms_200_silver_view_new_data')

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 60, Finished, Available, Finished)

In [55]:
create_silver_table = f"""
    CREATE TABLE lms_200_silver (
    Student_ID STRING,
    Name STRING,
    Age INTEGER NOT NULL,
    Gender STRING NOT NULL,
    Grade_Level STRING,
    Course_ID STRING,
    Course_Name STRING,
    Enrollment_Date DATE,
    Completion_Date DATE,
    Status STRING NOT NULL,
    Final_Grade STRING NOT NULL,
    Attendance_Rate DOUBLE NOT NULL,
    Time_Spent_on_Course_hrs DOUBLE NOT NULL,
    Assignments_Completed INTEGER NOT NULL,
    Quizzes_Completed INTEGER NOT NULL,
    Forum_Posts INTEGER NOT NULL,
    Messages_Sent INTEGER NOT NULL,
    Quiz_Average_Score DOUBLE NOT NULL,
    Assignment_Scores STRING,
    Assignment_Average_Score DOUBLE NOT NULL,
    Project_Score DOUBLE NOT NULL,
    Extra_Credit DOUBLE NOT NULL,
    Overall_Performance DOUBLE NOT NULL,
    Feedback_Score DOUBLE NOT NULL,
    Parent_Involvement STRING NOT NULL,
    Demographic_Group STRING NOT NULL,
    Internet_Access STRING NOT NULL,
    Learning_Disabilities STRING NOT NULL,
    Preferred_Learning_Style STRING NOT NULL,
    Language_Proficiency STRING NOT NULL,
    Participation_Rate STRING NOT NULL,
    Completion_Time_Days INTEGER,
    Performance_Score DOUBLE NOT NULL,
    Course_Completion_Rate STRING NOT NULL,
    Processing_Date DATE ); """

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 61, Finished, Available, Finished)

In [56]:
try:
    spark \
    .read \
    .table("ms_lms_lh_200_silver.lms_200_silver") \
    .createOrReplaceTempView('lms_200_silver_view')
except:
    spark.sql(create_silver_table)

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 62, Finished, Available, Finished)

In [57]:

merge_lms_200_silver = \
f"""     
MERGE INTO lms_200_silver as TARGET
USING lms_200_silver_view_new_data as SOURCE
ON TARGET.Student_ID = SOURCE.Student_ID AND TARGET.Course_ID = SOURCE.Course_ID

WHEN MATCHED THEN
                            UPDATE SET
                                TARGET.Name =                       SOURCE.Name,
                                TARGET.Age =                        SOURCE.Age,
                                TARGET.Gender =                     SOURCE.Gender,
                                TARGET.Grade_Level =                SOURCE.Grade_Level,
                                TARGET.Course_Name =                SOURCE.Course_Name,
                                TARGET.Enrollment_Date =            SOURCE.Enrollment_Date,
                                TARGET.Completion_Date =            SOURCE.Completion_Date,
                                TARGET.Status =                     SOURCE.Status,
                                TARGET.Final_Grade =                SOURCE.Final_Grade,
                                TARGET.Attendance_Rate =            SOURCE.Attendance_Rate,
                                TARGET.Time_Spent_on_Course_hrs =   SOURCE.Time_Spent_on_Course_hrs,
                                TARGET.Assignments_Completed =      SOURCE.Assignments_Completed,
                                TARGET.Quizzes_Completed =          SOURCE.Quizzes_Completed,
                                TARGET.Forum_Posts =                SOURCE.Forum_Posts,
                                TARGET.Messages_Sent =              SOURCE.Messages_Sent,
                                TARGET.Quiz_Average_Score =         SOURCE.Quiz_Average_Score,
                                TARGET.Assignment_Scores =          SOURCE.Assignment_Scores,
                                TARGET.Assignment_Average_Score =   SOURCE.Assignment_Average_Score,
                                TARGET.Project_Score =              SOURCE.Project_Score,
                                TARGET.Extra_Credit =               SOURCE.Extra_Credit,
                                TARGET.Overall_Performance =        SOURCE.Overall_Performance,
                                TARGET.Feedback_Score =             SOURCE.Feedback_Score,
                                TARGET.Parent_Involvement =         SOURCE.Parent_Involvement,
                                TARGET.Demographic_Group =          SOURCE.Demographic_Group,
                                TARGET.Internet_Access =            SOURCE.Internet_Access,
                                TARGET.Learning_Disabilities =      SOURCE.Learning_Disabilities,
                                TARGET.Preferred_Learning_Style =   SOURCE.Preferred_Learning_Style,
                                TARGET.Language_Proficiency =       SOURCE.Language_Proficiency,
                                TARGET.Participation_Rate =         SOURCE.Participation_Rate,
                                TARGET.Completion_Time_Days =       SOURCE.Completion_Time_Days,
                                TARGET.Performance_Score =          SOURCE.Performance_Score,
                                TARGET.Course_Completion_Rate =     SOURCE.Course_Completion_Rate,
                                TARGET.Processing_Date =            SOURCE.Processing_Date
WHEN NOT MATCHED THEN
                            INSERT                  
                            (
                                TARGET.Student_ID, 
                                TARGET.Name, 
                                TARGET.Age, 
                                TARGET.Gender, 
                                TARGET.Grade_Level, 
                                TARGET.Course_ID, 
                                TARGET.Course_Name, 
                                TARGET.Enrollment_Date, 
                                TARGET.Completion_Date, 
                                TARGET.Status, 
                                TARGET.Final_Grade, 
                                TARGET.Attendance_Rate, 
                                TARGET.Time_Spent_on_Course_hrs, 
                                TARGET.Assignments_Completed, 
                                TARGET.Quizzes_Completed,
                                TARGET.Forum_Posts, 
                                TARGET.Messages_Sent, 
                                TARGET.Quiz_Average_Score, 
                                TARGET.Assignment_Scores, 
                                TARGET.Assignment_Average_Score, 
                                TARGET.Project_Score,
                                TARGET.Extra_Credit, 
                                TARGET.Overall_Performance, 
                                TARGET.Feedback_Score, 
                                TARGET.Parent_Involvement, 
                                TARGET.Demographic_Group, 
                                TARGET.Internet_Access,
                                TARGET.Learning_Disabilities, 
                                TARGET.Preferred_Learning_Style, 
                                TARGET.Language_Proficiency, 
                                TARGET.Participation_Rate, 
                                TARGET.Completion_Time_Days,
                                TARGET.Performance_Score, 
                                TARGET.Course_Completion_Rate, 
                                TARGET.Processing_Date
                            )
                            VALUES 
                            (
                                SOURCE.Student_ID,
                                SOURCE.Name, 
                                SOURCE.Age, 
                                SOURCE.Gender, 
                                SOURCE.Grade_Level, 
                                SOURCE.Course_ID, 
                                SOURCE.Course_Name, 
                                SOURCE.Enrollment_Date, 
                                SOURCE.Completion_Date, 
                                SOURCE.Status, 
                                SOURCE.Final_Grade, 
                                SOURCE.Attendance_Rate, 
                                SOURCE.Time_Spent_on_Course_hrs, 
                                SOURCE.Assignments_Completed, 
                                SOURCE.Quizzes_Completed, 
                                SOURCE.Forum_Posts, 
                                SOURCE.Messages_Sent, 
                                SOURCE.Quiz_Average_Score, 
                                SOURCE.Assignment_Scores, 
                                SOURCE.Assignment_Average_Score, 
                                SOURCE.Project_Score, 
                                SOURCE.Extra_Credit, 
                                SOURCE.Overall_Performance, 
                                SOURCE.Feedback_Score, 
                                SOURCE.Parent_Involvement, 
                                SOURCE.Demographic_Group, 
                                SOURCE.Internet_Access, 
                                SOURCE.Learning_Disabilities, 
                                SOURCE.Preferred_Learning_Style, 
                                SOURCE.Language_Proficiency, 
                                SOURCE.Participation_Rate, 
                                SOURCE.Completion_Time_Days, 
                                SOURCE.Performance_Score, 
                                SOURCE.Course_Completion_Rate,
                                SOURCE.Processing_Date
                            )
"""
                        

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 63, Finished, Available, Finished)

In [58]:
spark.sql(merge_lms_200_silver).show()

StatementMeta(, ec049dcc-d7cd-4e69-af3d-cdfae228ef53, 64, Finished, Available, Finished)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|               11|               0|               0|               11|
+-----------------+----------------+----------------+-----------------+

